# Fine Tune Small Language Model for Manim Code Generation
## Data Preparation

In [48]:
# Load data
from datasets import load_dataset

dataset = load_dataset("Edoh/manim_python")

In [49]:
# Load model tokenizer
from transformers import GPT2Tokenizer

model_name = "openai-community/gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [50]:
# Preprocess the data for fine tuning
def preprocess_data(examples):
    inputs = [
        f"Instruction: {instr}\nOutput: {out}{tokenizer.eos_token}"
        for instr, out in zip(examples["instruction"], examples["output"])
    ]

    tokenized = tokenizer(inputs, truncation=True, max_length=512, padding="max_length")

    # Build labels manually: keep real tokens, mask padding to -100.
    # DataCollatorForLanguageModeling would mask ALL eos_token positions (including
    # the one at the end of the output), so we handle it ourselves instead.
    labels = []
    for input_ids, attention_mask in zip(tokenized["input_ids"], tokenized["attention_mask"]):
        label = [tok if mask == 1 else -100 for tok, mask in zip(input_ids, attention_mask)]
        labels.append(label)
    tokenized["labels"] = labels

    return tokenized

tokenized_datasets = dataset.map(
    preprocess_data,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Map:   0%|          | 0/51 [00:00<?, ? examples/s]

## Hyperparamter Search

In [51]:
from transformers import GPT2LMHeadModel

def model_init():
    return GPT2LMHeadModel.from_pretrained(model_name)

In [52]:
# Configure training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gpt2-manim-python-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

In [53]:
# Evaluation data split
train_val_split = tokenized_datasets["train"].train_test_split(test_size=0.1)
tokenized_datasets["train"] = train_val_split["train"]
tokenized_datasets["validation"] = train_val_split["test"]

In [54]:
# Initialize the Trainer with arguments and tokenized training and eval data
from transformers import (
   Trainer,
   EarlyStoppingCallback,
   default_data_collator,
)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [55]:
# Hyperparameter search space function
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [2, 4, 8]),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.3),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "warmup_steps": trial.suggest_int("warmup_steps", 0, 500),
        "gradient_accumulation_steps": trial.suggest_categorical("gradient_accumulation_steps", [1, 2, 4]),
    }

In [56]:
# best_run = trainer.hyperparameter_search(
#     direction="minimize",
#     backend="optuna",
#     n_trials=10,
#     hp_space=hp_space,
#     compute_objective=lambda metrics: metrics["eval_loss"],
# )

# Save best hyperparameters to a file
# import json
#
# best_hp = {
#     "run_id": best_run.run_id,
#     "objective": best_run.objective,
#     "hyperparameters": best_run.hyperparameters,
# }
#
# with open("saved/best_hyperparameters.json", "w") as f:
#     json.dump(best_hp, f, indent=2)
#
# print(f"Best run: {best_run.run_id}")
# print(f"Eval loss: {best_run.objective:.4f}")
# print(f"Hyperparameters: {json.dumps(best_run.hyperparameters, indent=2)}")

In [57]:
# Load saved hyperparameters (skip the search cell above if using this)
import json
from transformers.trainer_utils import BestRun

with open("saved/best_hyperparameters.json", "r") as f:
    saved = json.load(f)

best_run = BestRun(
    run_id=saved["run_id"],
    objective=saved["objective"],
    hyperparameters=saved["hyperparameters"],
)

print(f"Loaded best run: {best_run.run_id}, eval loss: {best_run.objective:.4f}")

Loaded best run: 6, eval loss: 0.1412


In [58]:
print(best_run)

BestRun(run_id='6', objective=0.14122234284877777, hyperparameters={'learning_rate': 6.325134781584482e-05, 'per_device_train_batch_size': 2, 'weight_decay': 0.26190960900619836, 'num_train_epochs': 6, 'warmup_steps': 365, 'gradient_accumulation_steps': 2}, run_summary=None)


In [59]:
# Configure trainer with Hyper parameters
for key, value in best_run.hyperparameters.items():
    setattr(training_args, key, value)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets.get("validation"),
    processing_class=tokenizer,
    data_collator=default_data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [60]:
trainer.train()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.239001,0.327749
2,0.423264,0.204434
3,0.207710,0.162384
4,0.176112,0.147906
5,0.148194,0.135033
6,0.121928,0.134600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/joewilkinson/Projects/scratchllm/env-llm/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=810, training_loss=0.4607998777318884, metrics={'train_runtime': 1233.5584, 'train_samples_per_second': 2.622, 'train_steps_per_second': 0.657, 'total_flos': 845018431488000.0, 'train_loss': 0.4607998777318884, 'epoch': 6.0})

In [61]:
# Save fine-tuned model and tokenizer
trainer.save_model("./saved/gpt2-manim-python-finetuned")
tokenizer.save_pretrained("./saved/gpt2-manim-python-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved/gpt2-manim-python-finetuned/tokenizer_config.json',
 './saved/gpt2-manim-python-finetuned/tokenizer.json')

## Test Fine-Tuned Model

In [62]:
import torch

model_dir = "./saved/gpt2-manim-python-finetuned"
tokenizer = GPT2Tokenizer.from_pretrained(model_dir)
model = GPT2LMHeadModel.from_pretrained(model_dir)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

model.to(device)
print(f"Using device: {device}")

model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Using device: mps


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [63]:
def generate_output(
        instruction,
        max_length=150,
        num_beams=5,
        repetition_penalty=1.2,
):
    prompt = f"Instruction: {instruction}\nOutput:"
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    generated_ids = model.generate(
        input_ids,
        max_length=max_length,
        num_beams=num_beams,
        repetition_penalty=repetition_penalty,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True,
    )

    generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    output_start = generated_text.find("Output:")
    if output_start != -1:
        output_text = generated_text[output_start + len("Output:"):].strip()
    else:
        output_text = generated_text.strip()

    return output_text

In [64]:
# Generate outputs for the test set and save to CSV
import csv

output_csv = "saved/gpt2_manim_python_test_outputs.csv"
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(
        csvfile,
        fieldnames=["instruction", "reference_output", "generated_output"],
    )
    writer.writeheader()

    for i, example in enumerate(dataset["test"]):
        instruction = example["instruction"]
        reference_output = example["output"]

        generated_output = generate_output(instruction)

        writer.writerow({
            "instruction": instruction,
            "reference_output": reference_output,
            "generated_output": generated_output,
        })
        print(f"[{i+1}/{len(dataset['test'])}] {instruction[:60]}...")

print(f"\nResults saved to {output_csv}")

[1/51] Rotate the pentagon by 90 degrees counterclockwise over 2 se...
[2/51] Create a rectangle with width 4 and height 2, centered at (1...
[3/51] Create a VGroup and add a circle and a triangle to it....
[4/51] Rotate the VGroup by 45 degrees clockwise over 1.5 seconds....
[5/51] Create a regular hexagon with side length 2 and color it yel...
[6/51] Move the hexagon to the left by 2 units over 1 second....
[7/51] Create a regular heptagon with side length 3 and color it gr...
[8/51] Rotate the heptagon by 60 degrees counterclockwise over 2 se...
[9/51] Create a line segment from (-3, 0) to (3, 0) and color it pu...
[10/51] Move the line segment upwards by 2 units over 1 second....
[11/51] Create an equilateral triangle with side length 3 and color ...
[12/51] Create a circle with radius 3 and color it blue....
[13/51] Move the circle to the right by 2 units over 1 second....
[14/51] Create a square with side length 4 and color it red....
[15/51] Scale the square by a factor of 2 in 

## Automated Evaluation

In [65]:
# Reformat single-line condensed Manim code to properly indented multi-line Python.
# The Edoh/manim_python dataset stores code on a single line, which isn't valid Python
# (e.g. "class X(Scene): def construct(self): pass" is a SyntaxError).
# This uses the stdlib tokenize module to split at statement boundaries.
import tokenize
import token as token_mod
import io

_NO_SPACE_BEFORE = {")", "]", "}", ",", ":", "."}
_NO_SPACE_AFTER = {"(", "[", "{", ".", "~"}

def _is_keyword_arg(tok_strings, eq_index):
    depth = 0
    for j in range(eq_index):
        if tok_strings[j] in ("(", "[", "{"):
            depth += 1
        elif tok_strings[j] in (")", "]", "}"):
            depth -= 1
    return depth > 0

def _reconstruct_tokens(tok_strings):
    if not tok_strings:
        return ""
    parts = [tok_strings[0]]
    for i in range(1, len(tok_strings)):
        prev = tok_strings[i - 1]
        curr = tok_strings[i]
        if curr in _NO_SPACE_BEFORE or prev in _NO_SPACE_AFTER:
            parts.append(curr)
        elif curr == "(" and prev not in ("=", ",", "return", "and", "or", "not", "in", "if", "else"):
            parts.append(curr)
        elif curr == "-" and prev in ("(", ",", "="):
            parts.append(curr)
        elif prev == "-" and tok_strings[i - 2:i - 1] in [["("], [","], ["="]]:
            parts.append(curr)
        elif curr == "=" and _is_keyword_arg(tok_strings, i):
            parts.append(curr)
        elif prev == "=" and i >= 2 and _is_keyword_arg(tok_strings, i - 1):
            parts.append(curr)
        else:
            parts.append(" " + curr)
    result = "".join(parts)
    result = result.replace("import*", "import *")
    return result

def reformat_condensed_manim(code_str):
    if "\n" in code_str:
        first_line = code_str.split("\n")[0]
    else:
        first_line = code_str

    try:
        tokens = list(tokenize.generate_tokens(io.StringIO(first_line).readline))
    except tokenize.TokenError:
        return code_str

    meaningful = [
        t for t in tokens
        if t.type not in (token_mod.NEWLINE, token_mod.ENDMARKER, token_mod.NL,
                          token_mod.COMMENT, token_mod.ENCODING)
    ]

    if not meaningful:
        return code_str

    splits = []
    paren_depth = 0
    awaiting_body = False
    in_method_body = False

    for i, tok in enumerate(meaningful):
        if tok.type == token_mod.OP and tok.string in ("(", "[", "{"):
            paren_depth += 1
        elif tok.type == token_mod.OP and tok.string in (")", "]", "}"):
            paren_depth = max(0, paren_depth - 1)

        if paren_depth > 0:
            continue

        if tok.type == token_mod.NAME and tok.string == "class":
            splits.append((i, 0))
            in_method_body = False
            awaiting_body = False
        elif tok.type == token_mod.NAME and tok.string == "def":
            splits.append((i, 1))
            in_method_body = False
            awaiting_body = True
        elif awaiting_body and tok.type == token_mod.OP and tok.string == ":":
            awaiting_body = False
            in_method_body = True
            for j in range(i + 1, len(meaningful)):
                if meaningful[j].type not in (token_mod.NEWLINE, token_mod.NL):
                    splits.append((j, 2))
                    break
        elif in_method_body and tok.type == token_mod.NAME:
            prev_tok = meaningful[i - 1] if i > 0 else None
            if prev_tok and prev_tok.type == token_mod.OP and prev_tok.string in (")", "]", "}"):
                splits.append((i, 2))

    if not splits:
        return code_str

    if splits[0][0] > 0:
        splits.insert(0, (0, 0))

    splits = sorted(set(splits), key=lambda x: x[0])

    indent_map = {0: "", 1: "    ", 2: "        "}
    lines = []
    for seg_idx, (start, indent_level) in enumerate(splits):
        end = splits[seg_idx + 1][0] if seg_idx + 1 < len(splits) else len(meaningful)
        seg_tokens = [meaningful[j].string for j in range(start, end)]
        if not seg_tokens:
            continue
        indent = indent_map.get(indent_level, "        ")
        lines.append(indent + _reconstruct_tokens(seg_tokens))

    result_lines = []
    for i, line in enumerate(lines):
        if line.strip().startswith("class ") and i > 0:
            result_lines.append("")
        result_lines.append(line)

    result = "\n".join(result_lines)

    try:
        ast.parse(result)
    except SyntaxError:
        return code_str

    return result

In [66]:
# Validate reformatter against all reference outputs
ref_pass = 0
for example in dataset["test"]:
    reformatted = reformat_condensed_manim(example["output"])
    try:
        ast.parse(reformatted)
        ref_pass += 1
    except SyntaxError:
        pass

print(f"Reference outputs passing syntax after reformatting: {ref_pass}/{len(dataset['test'])}")
assert ref_pass == len(dataset["test"]), "Reformatter should handle all reference outputs!"

Reference outputs passing syntax after reformatting: 51/51


In [67]:
# Check Python syntax
import ast

def is_syntax_valid(code_str):
    try:
        ast.parse(code_str)
        return True, ""
    except SyntaxError as e:
        return False, str(e)

In [68]:
# Static code analysis
from ast import NodeVisitor

class ManimCodeAnalyzer(NodeVisitor):
    def __init__(self):
        self.imports_manim = False
        self.scene_subclass_names = []
        self.play_calls = 0
        self.create_calls = 0
        self.errors = []

    def visit_Import(self, node):
        for alias in node.names:
            if alias.name == "manim":
                self.imports_manim = True
        self.generic_visit(node)

    def visit_ImportFrom(self, node):
        if node.module and node.module.startswith("manim"):
            self.imports_manim = True
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        for base in node.bases:
            if isinstance(base, ast.Name) and base.id == "Scene":
                self.scene_subclass_names.append(node.name)
            elif isinstance(base, ast.Attribute):
                if base.attr == "Scene":
                    self.scene_subclass_names.append(node.name)
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Attribute):
            if (
                isinstance(node.func.value, ast.Name)
                    and node.func.value.id == "self"
                    and node.func.attr == "play"
            ):
                self.play_calls += 1

        if isinstance(node.func, ast.Name) and node.func.id == "Create":
            self.create_calls += 1

        self.generic_visit(node)

In [69]:
def analyze_manim_code(code_str):
    analyzer = ManimCodeAnalyzer()
    try:
        tree = ast.parse(code_str)
    except SyntaxError as e:
        return {
            "syntax_valid": False,
            "syntax_error": str(e),
            "imports_manim": False,
            "scene_subclass_names": [],
            "play_calls": 0,
            "create_calls": 0,
        }

    analyzer.visit(tree)
    return {
        "syntax_valid": True,
        "syntax_error": None,
        "imports_manim": analyzer.imports_manim,
        "scene_subclass_names": analyzer.scene_subclass_names,
        "play_calls": analyzer.play_calls,
        "create_calls": analyzer.create_calls,
    }

In [70]:
# Execute generated Manim code via the CLI and check if rendering succeeds
import os
import tempfile
import subprocess

def evaluate_manim_code(code_str, scene_class_name="CustomScene"):
    with tempfile.TemporaryDirectory(dir="./saved") as tmpdir:
        code_path = os.path.join(tmpdir, "generated_scene.py")
        with open(code_path, "w") as f:
            f.write(code_str)

        cmd = [
            "manim",
            "-ql",
            code_path,
            scene_class_name,
        ]

        try:
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
            success = result.returncode == 0
            output = result.stdout + "\n" + result.stderr
        except subprocess.TimeoutExpired:
            success = False
            output = "Timeout expired during rendering."

    return success, output

In [71]:
# Full evaluation pipeline: reformat -> syntax check -> static analysis -> render execution
import csv

output_csv = "saved/gpt2_manim_evaluation_results.csv"
fieldnames = [
    "instruction",
    "reference_output",
    "generated_output",
    "formatted_output",
    "syntax_valid",
    "syntax_error",
    "imports_manim",
    "scene_subclass_names",
    "play_calls",
    "create_calls",
    "render_success",
    "render_output",
]

results = []
with open(output_csv, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    for i, example in enumerate(dataset["test"]):
        instruction = example["instruction"]
        reference_output = example["output"]
        generated_output = generate_output(instruction)
        formatted_output = reformat_condensed_manim(generated_output)

        # Step 1: Syntax check (on reformatted code)
        syntax_valid, syntax_error = is_syntax_valid(formatted_output)

        # Step 2: Static analysis
        analysis = analyze_manim_code(formatted_output)

        # Step 3: Render execution (only if syntax is valid and a Scene class was found)
        scene_names = analysis["scene_subclass_names"]
        if syntax_valid and scene_names:
            render_success, render_output = evaluate_manim_code(
                formatted_output, scene_class_name=scene_names[0]
            )
        else:
            render_success = False
            render_output = "Skipped: " + (
                f"syntax error ({syntax_error})" if not syntax_valid
                else "no Scene subclass found"
            )

        row = {
            "instruction": instruction,
            "reference_output": reference_output,
            "generated_output": generated_output,
            "formatted_output": formatted_output,
            "syntax_valid": syntax_valid,
            "syntax_error": syntax_error,
            "imports_manim": analysis["imports_manim"],
            "scene_subclass_names": ", ".join(scene_names),
            "play_calls": analysis["play_calls"],
            "create_calls": analysis["create_calls"],
            "render_success": render_success,
            "render_output": render_output,
        }
        results.append(row)
        writer.writerow(row)

        status = "PASS" if render_success else "FAIL"
        print(f"[{i+1}/{len(dataset['test'])}] {status} | syntax={syntax_valid} scene={bool(scene_names)} render={render_success} | {instruction[:50]}...")

print(f"\nResults saved to {output_csv}")

[1/51] PASS | syntax=True scene=True render=True | Rotate the pentagon by 90 degrees counterclockwise...
[2/51] PASS | syntax=True scene=True render=True | Create a rectangle with width 4 and height 2, cent...
[3/51] FAIL | syntax=True scene=True render=False | Create a VGroup and add a circle and a triangle to...
[4/51] FAIL | syntax=True scene=True render=False | Rotate the VGroup by 45 degrees clockwise over 1.5...
[5/51] PASS | syntax=True scene=True render=True | Create a regular hexagon with side length 2 and co...
[6/51] PASS | syntax=True scene=True render=True | Move the hexagon to the left by 2 units over 1 sec...
[7/51] PASS | syntax=True scene=True render=True | Create a regular heptagon with side length 3 and c...
[8/51] PASS | syntax=True scene=True render=True | Rotate the heptagon by 60 degrees counterclockwise...
[9/51] FAIL | syntax=True scene=True render=False | Create a line segment from (-3, 0) to (3, 0) and c...
[10/51] FAIL | syntax=True scene=True render=False |

In [72]:
# Summary statistics
total = len(results)
syntax_pass = sum(1 for r in results if r["syntax_valid"])
has_scene = sum(1 for r in results if r["scene_subclass_names"])
imports_manim = sum(1 for r in results if r["imports_manim"])
render_pass = sum(1 for r in results if r["render_success"])

print(f"Total test examples:   {total}")
print(f"Valid Python syntax:   {syntax_pass}/{total} ({100*syntax_pass/total:.1f}%)")
print(f"Imports manim:         {imports_manim}/{total} ({100*imports_manim/total:.1f}%)")
print(f"Has Scene subclass:    {has_scene}/{total} ({100*has_scene/total:.1f}%)")
print(f"Render success:        {render_pass}/{total} ({100*render_pass/total:.1f}%)")

Total test examples:   51
Valid Python syntax:   51/51 (100.0%)
Imports manim:         51/51 (100.0%)
Has Scene subclass:    51/51 (100.0%)
Render success:        29/51 (56.9%)
